# Clase 5 — Consultar una API con `requests`

Una **API** permite que un programa solicite datos o funciones a otro sistema. En esta notebook consultaremos Open-Meteo, una API meteorológica que no requiere clave para este ejemplo.

## 1. Importar la biblioteca

`requests` no forma parte de la biblioteca estándar. Debe instalarse en el entorno de la clase:

```bash
python -m pip install requests
```

In [4]:
%pip install requests
# Importamos la librería requests para hacer peticiones HTTP a una API.

# Gracias a ella podemos consultar datos desde internet desde Python.
import requests

Note: you may need to restart the kernel to use updated packages.


## 2. Definir la consulta

La URL identifica el servicio. El diccionario `parametros` contiene los datos enviados: ubicación, fechas y variables requeridas.

In [6]:
# La URL apunta al servicio de Open-Meteo que devuelve datos históricos del clima.
URL: str = "https://archive-api.open-meteo.com/v1/archive"

# En 'parametros' guardamos la información que pedimos a la API:
# latitud, longitud, fecha de inicio, fecha de fin, variables climáticas y zona horaria.
parametros: dict = {
    "latitude": -34.6037,
    "longitude": -58.3816,
    "start_date": "2026-08-01",
    "end_date": "2026-08-14",
    "daily": (
        "temperature_2m_max,"
        "temperature_2m_min,"
        "precipitation_sum"

    ),"timezone": "America/Argentina/Buenos_Aires",}
    

## 3. Realizar la petición y controlar errores

- `timeout=20` evita que el programa espere indefinidamente.
- `raise_for_status()` produce una excepción ante respuestas HTTP erróneas.
- `.json()` transforma la respuesta en un diccionario de Python.

Si no hay conexión, usaremos el CSV local en las notebooks siguientes.

In [7]:
import requests
datos_api: dict | None = None

try:
    respuesta = requests.get(
        URL,
        params=parametros,
        timeout=60
    )

    respuesta.raise_for_status()

    datos_api = respuesta.json()

    print("Consulta realizada correctamente.")
    print("Código HTTP:", respuesta.status_code)

except requests.RequestException as error:
    print("No fue posible consultar la API.")
    print("Detalle:", error)



Consulta realizada correctamente.
Código HTTP: 200


## 4. Explorar la respuesta

La respuesta vuelve a conectar APIs con diccionarios y listas. La clave `daily` contiene las series diarias.

In [10]:
# Solo entramos si la API respondió correctamente.
if datos_api is not None:
    # Mostramos las claves principales del diccionario JSON.
    print("Claves principales:", datos_api.keys())

    print("Primeras fechas:", datos_api["daily"]["time"][:3])

    # 'daily' es una sección que contiene los datos diarios del clima.    # Muestra las primeras fechas disponibles para ver cómo están estructurados los datos.

    print("Claves de daily:", datos_api["daily"].keys())

Claves principales: dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
Primeras fechas: ['2026-08-01', '2026-08-02', '2026-08-03']
Claves de daily: dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum'])


## 5. Transformar los datos

Crearemos una lista de diccionarios: un diccionario por día.

In [15]:
# Vamos a convertir la respuesta de la API en una lista más simple de manejar.
registros: list[dict] = []

if datos_api is not None:
    # 'daily' tiene todas las series diarias de la respuesta.
    diarios: dict = datos_api["daily"]

    # Recorremos cada fecha y tomamos los valores del mismo índice en cada lista.
    for indice, fecha in enumerate(diarios["time"]):
        registro: dict = {
            "fecha": fecha,
            "temperatura_maxima": diarios["temperature_2m_max"][indice],
            "temperatura_minima": diarios["temperature_2m_min"][indice],
            "precipitacion": diarios["precipitation_sum"][indice],
        }
        registros.append(registro)

print("Primer registro:", registros[0] if registros else "Sin datos")

# Mostramos cuántos registros generamos y el primer ejemplo.print("Cantidad de registros:", len(registros))

Primer registro: {'fecha': '2026-08-01', 'temperatura_maxima': 18.5, 'temperatura_minima': 12.2, 'precipitacion': 8.4}


## 6. Guardar la respuesta como JSON

Guardamos la respuesta original para poder revisarla sin hacer otra consulta.

In [9]:
# Importamos json para guardar la respuesta en formato JSON y Path para trabajar con rutas.
import json
from pathlib import Path

if datos_api is not None:
    # Definimos la ruta donde queremos guardar el archivo JSON dentro de la carpeta datos.
    ruta_json: Path = Path("datos") / "respuesta_open_meteo.json"
    ruta_json.parent.mkdir(exist_ok=True)

    # Guardamos la respuesta original en un archivo para poder revisarla luego sin consultar la API otra vez.
    with ruta_json.open("w", encoding="utf-8") as archivo:

        json.dump(datos_api, archivo, indent=4, ensure_ascii=False)    
        print(f"Archivo guardado en: {ruta_json}")


Archivo guardado en: datos/respuesta_open_meteo.json


# Ejercicio

Mostrá solamente los días cuya precipitación sea mayor que `0`. Para cada uno, imprimí fecha y milímetros registrados.

In [18]:
# En este ejercicio podés filtrar los días con precipitación mayor a 0.
# Para cada uno, mostrás la fecha y la cantidad de milímetros registrados.
def dias_con_precipitacion(registros: list[dict]) -> list[dict]:
    """
    Filtra los días con precipitación mayor a 0 y devuelve una lista de diccionarios
    con la fecha y la cantidad de milímetros registrados.
    """
    dias_filtrados: list[dict] = []

    for registro in registros:
        if registro["precipitacion"] > 0:
            dias_filtrados.append({
                "fecha": registro["fecha"],
                "precipitacion": registro["precipitacion"]
            })

    return dias_filtrados

# Escribí aquí tu solución.

dias_con_lluvia = dias_con_precipitacion(registros)
print("Días con precipitación mayor a 0:")
for dia in dias_con_lluvia:
    print(f"Fecha: {dia['fecha']}, Precipitación: {dia['precipitacion']} mm")   

Días con precipitación mayor a 0:
Fecha: 2026-08-01, Precipitación: 8.4 mm
Fecha: 2026-08-03, Precipitación: 2.4 mm
Fecha: 2026-08-05, Precipitación: 0.3 mm
Fecha: 2026-08-06, Precipitación: 23.6 mm
Fecha: 2026-08-13, Precipitación: 2.1 mm
Fecha: 2026-08-14, Precipitación: 0.2 mm
